# HelpBee v0.1.0 — 단일 스테이지 YOLO 학습 (AI Hub 71667)

> **대상 환경**: Elice 클라우드 A100 80GB · 웹 JupyterLab
> **아키텍처**: 단일 스테이지 3-class 검출 (`bee_normal` / `bee_with_varroa` / `bee_other_disease`), YOLOv11s @ imgsz 640
> **결정 근거**: `docs/01-development/adr/ADR-0001-yolo-engine-architecture.md`

기존 학습 코드(`training/train.py`, `eval.py`, `data/*`)를 **그대로 호출**하는 오케스트레이터입니다.

### 실행 순서
0. 사전 준비 → 1. 환경/저장소/의존성 → 2. 데이터 다운로드 → 3. 변환·golden·split → 4. 학습 → 5. 평가 → 6. ONNX → 7. 결과 보존/(선택) S3

### ⚠️ 주의
- **GPU 시간 과금** — 셀을 위에서부터 하나씩 확인하며 실행하세요.
- **71667 용량** — 전체 312,000장은 수백 GB(+병합용 2~3배 디스크). v0.1.0은 **filekey로 일부만** 받아 `LIMIT`으로 변환을 권장 (전체 다운로드는 명시적 opt-in).
- **라이선스 (미해결, ADR §8)** — 71667은 상용·내국인 제약. 상용 SaaS 학습 가부는 법무 확인 필요(내부 연구용 전제).
- **세션 휘발** — 종료 시 디스크가 날아갈 수 있으니 마지막 "결과 보존" 셀로 가중치를 꼭 내려받으세요.


## 0. 사전 준비 (수동, 1회)
1. **AI Hub API 키 발급** — [aihub.or.kr](https://www.aihub.or.kr) 로그인(내국인 계정) → 마이페이지 → API Key 발급(UUID 형식, 이메일 수신).
2. **데이터셋 신청 승인** — 71667 `꿀벌 질병 진단 이미지 데이터` 활용 신청 승인 필요.
3. (선택) **W&B 키**, **AWS 자격증명**(S3 업로드 시).


## 1. 환경 · 저장소 · 의존성

In [ ]:
# GPU / 파이썬 확인
!nvidia-smi
import sys; print("python", sys.version.split()[0])

In [ ]:
# 저장소 위치 자동 탐지 → 없으면 clone (repo PUBLIC). AI_ROOT를 전역으로 고정.
import os, subprocess
from pathlib import Path

REPO_URL = "https://github.com/hyunshu12/HelpBee.git"
# 교정된 config(yolo11s/640/3-class)가 있는 브랜치. develop 머지 후 "develop"로 바꾸세요.
BRANCH = "feature/ai-yolo-arch-decision"

def find_ai_root():
    here = Path.cwd().resolve()
    for d in [here, *here.parents]:
        if (d / "training" / "train.py").exists() and (d / "CLAUDE.md").exists():
            return d
    return None

ai_root = find_ai_root()
if ai_root is None:
    if not Path("HelpBee").exists():
        try:
            subprocess.run(["git", "clone", "--branch", BRANCH, "--single-branch", REPO_URL], check=True)
        except subprocess.CalledProcessError:
            print(f"⚠️ '{BRANCH}' clone 실패 → develop로 재시도")
            subprocess.run(["git", "clone", "--branch", "develop", "--single-branch", REPO_URL], check=True)
    ai_root = (Path("HelpBee") / "apps" / "ai").resolve()

os.chdir(ai_root)
AI_ROOT = Path.cwd().resolve()          # 이후 모든 절대경로의 기준
print("AI root (CWD):", AI_ROOT)
assert (AI_ROOT / "training" / "train.py").exists(), "training/train.py 를 찾지 못함 — 경로 확인"

In [ ]:
# 의존성 설치
# ⚠️ Elice A100 이미지에는 torch+CUDA가 사전설치됨 → torch는 건드리지 않고 나머지만 설치.
#    numpy 상한을 두지 않는다 (사전설치 torch가 numpy 2.x로 빌드된 경우 다운그레이드 시 ABI 깨짐).
%pip install -q "ultralytics>=8.3,<8.4" "albumentations>=1.4,<2.0" wandb "pandas>=2.1" "scikit-learn>=1.4" tqdm "pyyaml>=6.0" "numpy>=1.24"

import numpy, torch
print("numpy", numpy.__version__, "| torch", torch.__version__, "| CUDA:", torch.cuda.is_available())
if torch.cuda.is_available():
    print("GPU:", torch.cuda.get_device_name(0))
else:
    print("⚠️ CUDA 미인식 — torch가 CPU판이면 학습이 매우 느림. CUDA torch 재설치 필요.")
print("※ 만약 numpy가 다운그레이드됐다는 경고/ABI 오류가 보이면 커널을 재시작(Kernel > Restart)하고 이 셀부터 다시 실행하세요.")

## 2. 데이터 다운로드 (AI Hub 71667 · aihubshell)

In [ ]:
# aihubshell 설치 + 무결성 검증 (HTML/빈 파일 방어)
import os, subprocess
from pathlib import Path
subprocess.run(["curl", "-fsSL", "-o", "aihubshell", "https://api.aihub.or.kr/api/aihubshell.do"], check=True)
data = Path("aihubshell").read_bytes()
assert len(data) > 1024 and data[:2] == b"#!", "aihubshell 다운로드 실패(HTML/빈 파일?). 네트워크/URL 확인."
os.chmod("aihubshell", 0o755)
AIHUBSHELL = str((Path("aihubshell")).resolve())
print("aihubshell OK:", AIHUBSHELL)

In [ ]:
# 파일 목록 + filekey 확인 (list 모드는 API 키 불필요). 출력이 잘 보이도록 shell magic 사용.
!bash {AIHUBSHELL} -mode l -datasetkey 71667

In [ ]:
# AI Hub API 키 입력 (UUID 형식). 화면에 노출되지 않음.
import os, getpass
AIHUB_APIKEY = os.environ.get("AIHUB_APIKEY") or getpass.getpass("AI Hub API key: ")
assert AIHUB_APIKEY, "API 키가 비어 있음"

In [ ]:
# 데이터 다운로드 (다운로드 후 분할압축 자동 병합·해제)
# 권장: 위 목록에서 filekey를 골라 일부만 받기. 전체(312k, 수백 GB)는 DOWNLOAD_ALL=True로 명시 opt-in.
import os, shutil, subprocess
from pathlib import Path

FILEKEYS = ""             # 예: "51939,51940" (cell 위 목록 참고). 비우면 ↓ DOWNLOAD_ALL 확인.
DOWNLOAD_ALL = False      # True여야만 전체 다운로드 허용.

DATA_DIR = AI_ROOT / "training/datasets/aihub-71667"
DATA_DIR.mkdir(parents=True, exist_ok=True)

free_gb = shutil.disk_usage(str(DATA_DIR)).free / 1e9
print(f"여유 디스크: {free_gb:.1f} GB")
if not FILEKEYS and not DOWNLOAD_ALL:
    raise SystemExit("FILEKEYS에 filekey를 넣거나, 전체를 받으려면 DOWNLOAD_ALL=True 로 설정하세요.")
if DOWNLOAD_ALL and not FILEKEYS and free_gb < 800:
    raise SystemExit(f"전체 다운로드+병합 디스크 부족 추정(여유 {free_gb:.0f}GB). filekey 일부만 받으세요.")

cmd = ["bash", AIHUBSHELL, "-mode", "d", "-datasetkey", "71667", "-aihubapikey", AIHUB_APIKEY]
if FILEKEYS:
    cmd += ["-filekey", FILEKEYS]

print("다운로드 시작 (키·전체 cmd는 출력 생략)…")
try:
    subprocess.run(cmd, cwd=str(DATA_DIR), check=True)   # 실패해도 cmd(키 포함)를 노출하지 않음
except subprocess.CalledProcessError as e:
    raise SystemExit(f"aihubshell 다운로드 실패 (returncode={e.returncode}). 키/데이터 승인/디스크 확인.") from None
print("완료.")

In [ ]:
# ⚠️ 변환 전 레이아웃 확인 (aihubshell 산출 구조가 변환기 기대와 맞는지)
# aihub_to_yolo는 "02.라벨링데이터"→"01.원천데이터" 치환으로 이미지를 찾는다.
from pathlib import Path
jsons = list(DATA_DIR.rglob("*.json"))
jpgs = list(DATA_DIR.rglob("*.jpg"))
print(f"JSON 라벨: {len(jsons)}개 | JPG 이미지: {len(jpgs)}개")
assert jsons, "JSON 라벨 0건 — 다운로드/압축해제 또는 --source 경로 확인."
print("예시 JSON:", jsons[0])
print("예시 JPG :", jpgs[0] if jpgs else "없음")
print("\n디렉터리 구조 (라벨/원천 폴더명 확인):")
!find {str(DATA_DIR)} -maxdepth 4 -type d | head -40

## 3. YOLO 변환 → golden 추출 → train/val 분할

순서 주의: **golden을 split보다 먼저** 뽑고, golden 이미지를 allraw에서 제거해 train/golden 누수를 막습니다.
⚠️ `LIMIT`은 JSON을 정렬해 앞에서 N개를 자르므로 colony 001(71667의 75%) 편중이 더 심해질 수 있습니다 — golden 추출(3-2)의 농가/기기 다양성 경고를 꼭 확인하세요.

In [ ]:
# 3-1. AI Hub JSON → YOLO 형식 변환 (7-class → 3-class). LIMIT=None이면 전체.
import sys, subprocess
from pathlib import Path
LIMIT = 5000                       # v0.1.0 변환 이미지 수 (None=전체). A100 80GB면 더 키워도 됨.
ALLRAW = str(AI_ROOT / "training/datasets/varroa-v1-allraw")
cmd = [sys.executable, "-m", "training.data.aihub_to_yolo",
       "--source", str(DATA_DIR), "--output", ALLRAW, "--split", "all"]
if LIMIT is not None:
    cmd += ["--limit", str(LIMIT)]
subprocess.run(cmd, check=True)

In [ ]:
# 3-2. Golden holdout 추출 (응애 100 + 정상 200 = 300장, 농가≥3·기기≥2)
import sys, subprocess
from pathlib import Path
GOLDEN = str(AI_ROOT / "training/datasets/golden")
subprocess.run([sys.executable, "-m", "training.data.golden_holdout",
                "--input", ALLRAW, "--output", GOLDEN,
                "--n-varroa", "100", "--n-normal", "200",
                "--min-farms", "3", "--min-devices", "2"], check=True)

In [ ]:
# 3-3. ⚠️ golden 이미지를 allraw에서 제거 (train/golden 누수 방지; golden_holdout은 자동 제거 안 함)
import json
from pathlib import Path
allraw = Path(ALLRAW)
manifest = json.loads((Path(GOLDEN) / "manifest.json").read_text())
removed = 0
for entry in manifest:
    name = entry["image"]; stem = Path(name).stem
    for sub in ("all", "train"):
        img = allraw / "images" / sub / name
        lbl = allraw / "labels" / sub / f"{stem}.txt"
        if img.exists(): img.unlink(); removed += 1
        if lbl.exists(): lbl.unlink()
print(f"golden {removed}장을 allraw에서 제거 (누수 방지)")

In [ ]:
# 3-4. train/val 분할 (per_colony_time_block 자동 선택, data leakage 방지)
import sys, subprocess
VARROA_V1 = str(AI_ROOT / "training/datasets/varroa-v1")
subprocess.run([sys.executable, "-m", "training.data.split_strategy",
                "--input", ALLRAW, "--output", VARROA_V1,
                "--val-ratio", "0.2", "--seed", "42"], check=True)

In [ ]:
# 3-5. Ultralytics 경로 안전화 — dataset.yaml / golden data.yaml 의 path를 절대경로로
#      (상대 path는 Ultralytics datasets_dir 기준으로 해석되어 "dataset not found"가 날 수 있음)
#      ⚠️ 서버 클론의 yaml을 수정한다. git 커밋하지 말 것(절대경로는 로컬 전용).
import yaml
from pathlib import Path
def absolutize(yaml_rel, abs_dataset_dir):
    p = AI_ROOT / yaml_rel
    cfg = yaml.safe_load(p.read_text())
    cfg["path"] = str(Path(abs_dataset_dir).resolve())
    p.write_text(yaml.safe_dump(cfg, allow_unicode=True, sort_keys=False))
    assert Path(cfg["path"]).exists(), f"경로 없음: {cfg['path']}"
    print(f"{yaml_rel}: path → {cfg['path']}")
absolutize("training/configs/dataset.yaml", AI_ROOT / "training/datasets/varroa-v1")
absolutize("training/datasets/golden/data.yaml", AI_ROOT / "training/datasets/golden")

## 4. 학습

In [ ]:
# (선택) W&B 로깅 — 안 쓰면 이 셀 그대로(USE_WANDB=False) 두고 다음 셀 실행
import os, getpass
USE_WANDB = False
if USE_WANDB:
    os.environ["WANDB_API_KEY"] = os.environ.get("WANDB_API_KEY") or getpass.getpass("W&B API key: ")
    os.environ["WANDB_PROJECT"] = "helpbee-yolo"
    try:
        from ultralytics import settings
        settings.update({"wandb": True})
    except Exception as e:
        print("ultralytics wandb 설정 스킵:", e)
else:
    os.environ["WANDB_DISABLED"] = "true"
print("W&B:", "ON" if USE_WANDB else "OFF")

In [ ]:
# 학습 — config: yolo11s / imgsz 640 / 3-class (ADR-0001)
# A100 80GB이면 batch 64~128로 키워 학습 시간 단축 가능 (config 기본 16).
import sys, subprocess
subprocess.run([sys.executable, "-m", "training.train",
                "--config", "training/configs/yolo.yaml",
                "--name", "v0.1.0-baseline", "--batch", "64", "--device", "0"], check=True)

## 5. Golden 평가 (ADR §8 Q4 — v0.1.0 베이스라인 수치 확보)

In [ ]:
# golden 300장 평가 → mAP / varroa_recall(=bee_with_varroa recall) / infestation_rate MAE
import sys, subprocess
subprocess.run([sys.executable, "-m", "training.eval",
                "--weights", "runs/yolo/v0.1.0-baseline/weights/best.pt",
                "--golden", str(AI_ROOT / "training/datasets/golden/data.yaml"),
                "--imgsz", "640"], check=True)

In [ ]:
# 평가 JSON 확인 (eval_history에 커밋해 버전 비교 추적 권장)
import json
from pathlib import Path
ev = AI_ROOT / "runs/yolo/v0.1.0-baseline/weights/eval_golden.json"
print(json.dumps(json.loads(ev.read_text()), ensure_ascii=False, indent=2) if ev.exists()
      else "eval_golden.json 없음 — 평가 셀을 먼저 실행하세요.")

## 6. ONNX export (CPU 추론 서버용)

In [ ]:
from ultralytics import YOLO
YOLO("runs/yolo/v0.1.0-baseline/weights/best.pt").export(
    format="onnx", dynamic=True, simplify=True, imgsz=640)
print("ONNX export 완료 → runs/yolo/v0.1.0-baseline/weights/best.onnx")

## 7. 결과 보존 (필수) + (선택) S3 업로드
Elice 세션 종료 시 디스크가 사라질 수 있으니 **가중치를 반드시 내려받으세요**.

In [ ]:
# 7-1. 산출물 zip → JupyterLab 파일 브라우저에서 다운로드
import shutil
from pathlib import Path
w = AI_ROOT / "runs/yolo/v0.1.0-baseline/weights"
out = AI_ROOT / "helpbee_yolo_v0.1.0_artifacts"
out.mkdir(exist_ok=True)
for f in ("best.pt", "best.onnx", "eval_golden.json"):
    if (w / f).exists():
        shutil.copy2(w / f, out / f)
shutil.make_archive(str(out), "zip", str(out))
print(f"→ {out}.zip 생성. 왼쪽 파일 브라우저에서 우클릭 → Download.")

In [ ]:
# 7-2. (선택) S3 업로드 — s3://helpbee-models/yolo/v0.1.0/
UPLOAD_S3 = False
if UPLOAD_S3:
    import boto3, hashlib, json
    from pathlib import Path
    BUCKET, PREFIX = "helpbee-models", "yolo/v0.1.0"
    w = AI_ROOT / "runs/yolo/v0.1.0-baseline/weights"
    s3 = boto3.client("s3", region_name="ap-northeast-2")  # AWS 자격증명 사전 설정 필요
    sha = lambda p: hashlib.sha256(Path(p).read_bytes()).hexdigest()
    ev = json.loads((w / "eval_golden.json").read_text()) if (w / "eval_golden.json").exists() else {}
    metadata = {
        "stage": "single_stage_detector", "version": "v0.1.0", "model": "yolo11s", "imgsz": 640,
        "classes": ["bee_normal", "bee_with_varroa", "bee_other_disease"], "dataset": "aihub-71667",
        "golden_metrics": {k: ev.get(k) for k in ("mAP@0.5", "mAP@0.5:0.95", "varroa_recall", "infestation_rate_mae")},
        "sha256": {f: sha(w / f) for f in ("best.pt", "best.onnx") if (w / f).exists()}, "adr": "ADR-0001",
    }
    (w / "metadata.json").write_text(json.dumps(metadata, ensure_ascii=False, indent=2))
    for f in ("best.pt", "best.onnx", "metadata.json"):
        if (w / f).exists():
            s3.upload_file(str(w / f), BUCKET, f"{PREFIX}/{f}"); print("uploaded", f"s3://{BUCKET}/{PREFIX}/{f}")
else:
    print("S3 업로드 건너뜀 (UPLOAD_S3=False)")

## 마무리 / 다음 단계
- **산출물**: `runs/yolo/v0.1.0-baseline/weights/{best.pt, best.onnx, eval_golden.json}` → zip 다운로드 확인.
- **ADR §8 Q4 해소**: `eval_golden.json`의 mAP·`varroa_recall`·`infestation_rate_mae`를 `training/eval_history/v0.1.0.json`으로 커밋해 비교 기준선으로.
- **데이터 확장**: `LIMIT`↑(예: 50000) 재변환 후 `make train-extend NAME=v0.1.1-50k`.
- **다음 (ADR)**: v0.2.0 2-stage는 71488/Zenodo 라이선스 게이트 통과 시. OpenAI Vision이 MVP 상시 엔진이므로 이 YOLO는 게이트된 비차단 트랙.
